# Notebook 07 – FinBERT Distress Score (Branch 1: NLP)

**Objective:** Fine-tune FinBERT as a regression model to produce a continuous Distress Score (0.0–1.0) from customer chat logs. Score each chat turn independently to preserve the Sentiment Shift signal.

**Input:** `data/processed/sbdr_final_dataset.csv` (30K × 88)  
**Output:** `data/processed/07_with_distress_scores.csv` — adds 6 new columns  
**Model:** `models/finbert/best_model.pt`

---

## Architectural Decisions

**Regression, not classification:**  
The roadmap defines Branch 1 output as a continuous Distress Score (0.0–1.0). We map `distress_encoded` (0,1,2,3) → (0.0, 0.33, 0.67, 1.0) and fine-tune with MSE loss on a single regression head. This directly matches what XGBoost expects downstream.

**Score each turn separately — do not concatenate:**  
The docs list 'Sentiment Shift (change in tone over the last 3 messages)' as a key feature. Concatenating turns would destroy this signal. Each turn is scored independently, producing per-customer: `distress_turn_1`, `distress_turn_2`, `distress_turn_3`, `distress_avg`, `distress_max`, and `distress_shift` (turn3 − turn1).

**Same train/val split as BiLSTM:**  
Indices loaded from `models/bilstm/train_val_indices.npz` to ensure no cross-contamination at integration time.

**Two-stage fine-tuning data:**  
Primary: `pb_sentence` (Financial PhraseBank — teaches financial sentiment vocabulary).  
Secondary: `mh_statement` (Mental Health Sentiment — teaches emotional distress language).  
Both are combined into one fine-tuning dataset with distress-aligned regression targets.

---


In [ ]:
# Cell 1: Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModel
from scipy import stats

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.backends.mps.is_available():
    print('MPS (Apple Silicon) available')


In [ ]:
# Cell 2: Configuration

CONFIG = {
    # ----- File Paths -----
    'input_path':       '../../../data/processed/sbdr_final_dataset.csv',
    'output_path':      '../../../data/processed/07_with_distress_scores.csv',
    'model_dir':        '../../../models/finbert',
    'manifest_path':    '../../../models/finbert/finbert_manifest.json',
    'bilstm_split_path':'../../../models/bilstm/train_val_indices.npz',
    'chat_logs_path':    '../../../data/processed/sbdr_chat_logs.csv',

    # ----- FinBERT Model -----
    'pretrained_model': 'ProsusAI/finbert',
    'max_length': 128,

    # ----- Column Names -----
    'chat_cols':     ['chat_turn_1', 'chat_turn_2', 'chat_turn_3'],
    'text_cols':     ['pb_sentence', 'mh_statement'],
    'distress_col':  'distress_encoded',
    'distress_level_col': 'distress_level',

    # ----- Regression Target Mapping -----
    'distress_map': {0: 0.0, 1: 0.33, 2: 0.67, 3: 1.0},

    # ----- Scoring (zero-shot, no training needed) -----
    'batch_size': 64,

    # ----- Random Seed -----
    'seed': 42
}

np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])

# Force CPU — torch.cuda.is_available() returns True via ROCm but segfaults on this
# system (Radeon 8060S / gfx1100 iGPU, ROCm driver incompatibility).
DEVICE = torch.device('cpu')
print(f'Using device: {DEVICE}')

Path(CONFIG['model_dir']).mkdir(parents=True, exist_ok=True)
print('Configuration loaded.')

In [ ]:
# Cell 3: Load Data + Shared Split Indices
# sbdr_final_dataset.csv — full 88-col dataset (for fine-tuning text + distress labels)
# sbdr_chat_logs.csv     — chat-only file (customer_id, distress_level, chat_turn_1/2/3)
#                          Built specifically for FinBERT in A5. Use this for scoring.

print('Loading full dataset (fine-tuning labels + text)...')
df = pd.read_csv(CONFIG['input_path'])
print(f'Full dataset: {len(df):,} rows, {len(df.columns)} columns')

print('Loading chat logs (for scoring)...')
df_chats = pd.read_csv(CONFIG['chat_logs_path'])
print(f'Chat logs: {len(df_chats):,} rows, {len(df_chats.columns)} columns')
print(f'Chat columns: {list(df_chats.columns)}')

# Verify chat logs align with full dataset (same order, same customer_ids)
assert len(df_chats) == len(df), 'Row count mismatch between datasets!'
if 'customer_id' in df.columns and 'customer_id' in df_chats.columns:
    assert (df['customer_id'].values == df_chats['customer_id'].values).all(), \
        'customer_id mismatch between datasets!'
    print('customer_id alignment: OK')

required = CONFIG['text_cols'] + [CONFIG['distress_col'], CONFIG['distress_level_col']]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns in full dataset: {missing}')
print(f'All required columns found')

# Load shared train/val indices from BiLSTM (Notebook 06)
split_path = Path(CONFIG['bilstm_split_path'])
if split_path.exists():
    splits = np.load(split_path)
    train_idx = splits['train_idx']
    val_idx   = splits['val_idx']
    print(f'Loaded shared split: {len(train_idx):,} train / {len(val_idx):,} val')
else:
    print(f'WARNING: {split_path} not found. Generating new 80/20 split.')
    print('Run Notebook 06 first to ensure consistent splits across the pipeline.')
    n_total   = len(df)
    n_train   = int(0.8 * n_total)
    all_idx   = np.random.permutation(n_total)
    train_idx = all_idx[:n_train]
    val_idx   = all_idx[n_train:]

df['distress_target'] = df[CONFIG['distress_col']].map(CONFIG['distress_map']).astype('float32')
print(f'\nDistress target distribution:')
print(df['distress_target'].value_counts().sort_index())


In [ ]:
# Cell 4: Zero-shot mode — no fine-tuning corpus needed
# Pre-trained ProsusAI/finbert already understands financial sentiment vocabulary.
# We skip fine-tuning entirely and score directly from classifier probabilities.
#
# Distress score formula:
#   distress = P(negative) * 1.0 + P(neutral) * 0.5 + P(positive) * 0.0
#
# This gives a continuous score in [0, 1] where:
#   high P(negative) → score near 1.0 (high distress)
#   high P(neutral)  → score near 0.5 (moderate)
#   high P(positive) → score near 0.0 (low distress)

print('Zero-shot mode: skipping fine-tuning corpus preparation.')
print('FinBERT will be used as pre-trained (ProsusAI/finbert — financial sentiment).')
print()
print('Distress score formula:')
print('  distress = P(negative) × 1.0 + P(neutral) × 0.5 + P(positive) × 0.0')

In [ ]:
# Cell 5: Load FinBERT Pre-trained Classifier (Zero-shot mode)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

print('Loading FinBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['pretrained_model'])

print('Loading FinBERT classifier (pre-trained, zero-shot)...')
model = AutoModelForSequenceClassification.from_pretrained(CONFIG['pretrained_model'])
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model on: {DEVICE}')
print(f'Parameters: {n_params:,}')
print(f'Label mapping: {model.config.id2label}')

In [ ]:
# Cell 6: Zero-shot mode — training skipped
# Scores are derived directly from pre-trained FinBERT classifier probabilities.
# No gradient updates, no epochs, no train/val split needed.

best_val_loss = float('nan')
best_model_path = Path(CONFIG['model_dir']) / 'best_model.pt'
history = {'train_loss': [], 'val_loss': [], 'val_corr': []}

torch.save(model.state_dict(), best_model_path)
print('Zero-shot mode: no fine-tuning performed.')
print(f'Pre-trained model state saved to: {best_model_path}')

In [ ]:
# Cell 7: Zero-shot mode — no training curves to plot
print('Zero-shot mode: no training history.')
print('Score distribution plots will appear in Cell 10 after scoring.')

In [ ]:
# Cell 8: Score All 3 Chat Turns Per Customer (Zero-shot FinBERT)
# Maps classifier probabilities to a continuous distress score in [0, 1].
# Each turn scored independently to preserve the Sentiment Shift signal.

import torch.nn.functional as F

# Resolve label indices from model config
id2label = model.config.id2label   # e.g. {0: 'positive', 1: 'negative', 2: 'neutral'}
label2idx = {v.lower(): k for k, v in id2label.items()}
NEG_IDX = label2idx['negative']
NEU_IDX = label2idx['neutral']
POS_IDX = label2idx['positive']
print(f'Label indices — negative: {NEG_IDX}, neutral: {NEU_IDX}, positive: {POS_IDX}')
print(f'Full id2label: {id2label}')
print()


def score_texts(texts, model, tokenizer, device, max_length, batch_size=64):
    """Score texts via pre-trained FinBERT classifier.
    distress = P(negative)*1.0 + P(neutral)*0.5 + P(positive)*0.0
    """
    all_scores = []
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch_texts = [str(t) for t in texts[i:i + batch_size]]
        encoding = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].to(device)
        attn_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attn_mask)
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()

        scores = probs[:, NEG_IDX] * 1.0 + probs[:, NEU_IDX] * 0.5 + probs[:, POS_IDX] * 0.0
        all_scores.extend(scores)

    return np.array(all_scores, dtype='float32')


print('Scoring chat turns from sbdr_chat_logs.csv...')
turn_scores = {}
for col in CONFIG['chat_cols']:
    print(f'  Scoring {col}...')
    turn_scores[col] = score_texts(
        df_chats[col].values, model, tokenizer, DEVICE, CONFIG['max_length'])
    print(f'    mean={turn_scores[col].mean():.4f}, '
          f'std={turn_scores[col].std():.4f}, '
          f'range=[{turn_scores[col].min():.4f}, {turn_scores[col].max():.4f}]')

print('\nAll turns scored.')

In [ ]:
# Cell 9: Engineer Derived Distress Features
# distress_turn_1/2/3 — per-turn scores
# distress_avg         — mean across 3 turns
# distress_max         — worst (highest distress) turn
# distress_shift       — turn3 minus turn1 (positive = worsening sentiment)

df['distress_turn_1'] = turn_scores['chat_turn_1']
df['distress_turn_2'] = turn_scores['chat_turn_2']
df['distress_turn_3'] = turn_scores['chat_turn_3']

df['distress_avg']   = (df['distress_turn_1'] + df['distress_turn_2'] + df['distress_turn_3']) / 3
df['distress_max']   = df[['distress_turn_1', 'distress_turn_2', 'distress_turn_3']].max(axis=1)
df['distress_shift'] = df['distress_turn_3'] - df['distress_turn_1']

new_cols = ['distress_turn_1', 'distress_turn_2', 'distress_turn_3',
            'distress_avg', 'distress_max', 'distress_shift']

print('Derived distress features:')
for col in new_cols:
    print(f'  {col:>20s}: mean={df[col].mean():.4f}, std={df[col].std():.4f}, '
          f'range=[{df[col].min():.4f}, {df[col].max():.4f}]')

print(f'\nDistress shift stats:')
print(f'  Worsening (shift > 0):  {(df["distress_shift"] > 0).sum():,} customers')
print(f'  Stable    (shift ~ 0):  {((df["distress_shift"].abs()) < 0.05).sum():,} customers')
print(f'  Improving (shift < 0):  {(df["distress_shift"] < 0).sum():,} customers')


In [ ]:
# Cell 10: Validation — Distress Scores vs Distress Tiers

tier_order  = ['low', 'moderate', 'high', 'severe']
tier_colors = {'low': '#00b894', 'moderate': '#fdcb6e', 'high': '#e17055', 'severe': '#d63031'}
distress_levels = df[CONFIG['distress_level_col']].values

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: distress_avg by Tier
tier_scores = [df.loc[distress_levels == t, 'distress_avg'].values for t in tier_order]
bp = axes[0].boxplot(tier_scores, labels=tier_order, patch_artist=True,
                     showfliers=False, widths=0.6)
for patch, tier in zip(bp['boxes'], tier_order):
    patch.set_facecolor(tier_colors[tier])
    patch.set_alpha(0.8)
axes[0].set_xlabel('Distress Tier')
axes[0].set_ylabel('Distress Score (avg)')
axes[0].set_title('Average Distress Score by Tier')
axes[0].grid(True, alpha=0.3)

f_stat, p_value = stats.f_oneway(*tier_scores)
print(f'ANOVA (distress_avg across tiers): F={f_stat:.2f}, p={p_value:.2e}')
if p_value < 0.05:
    print('  -> Statistically significant separation across tiers')

# Panel 2: distress_shift by Tier
tier_shifts = [df.loc[distress_levels == t, 'distress_shift'].values for t in tier_order]
bp2 = axes[1].boxplot(tier_shifts, labels=tier_order, patch_artist=True,
                      showfliers=False, widths=0.6)
for patch, tier in zip(bp2['boxes'], tier_order):
    patch.set_facecolor(tier_colors[tier])
    patch.set_alpha(0.8)
axes[1].set_xlabel('Distress Tier')
axes[1].set_ylabel('Sentiment Shift (turn3 - turn1)')
axes[1].set_title('Sentiment Shift by Tier')
axes[1].axhline(0, color='grey', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

# Panel 3: Scatter — distress_avg vs distress_target
for tier in tier_order:
    mask = distress_levels == tier
    axes[2].scatter(df.loc[mask, 'distress_target'],
                    df.loc[mask, 'distress_avg'],
                    c=tier_colors[tier], label=tier, alpha=0.3, s=10)
axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect')
axes[2].set_xlabel('Target Distress Score')
axes[2].set_ylabel('Predicted Distress Score (avg)')
axes[2].set_title('Predicted vs Target')
axes[2].legend(markerscale=3)
axes[2].grid(True, alpha=0.3)

overall_corr = np.corrcoef(df['distress_target'], df['distress_avg'])[0, 1]
print(f'Overall correlation (target vs predicted avg): {overall_corr:.4f}')

plt.tight_layout()
plt.savefig(Path(CONFIG['model_dir']) / 'distress_validation.png', dpi=150)
plt.show()

print(f'\nPer-tier distress_avg:')
for tier in tier_order:
    mask = distress_levels == tier
    m = df.loc[mask, 'distress_avg'].mean()
    s = df.loc[mask, 'distress_avg'].std()
    print(f'  {tier:>10s}: {m:.4f} +/- {s:.4f}')


In [ ]:
# Cell 11: Save Output CSV

df_output = df.drop(columns=['distress_target'])
df_output.to_csv(CONFIG['output_path'], index=False)

new_cols = ['distress_turn_1', 'distress_turn_2', 'distress_turn_3',
            'distress_avg', 'distress_max', 'distress_shift']

print(f'Output saved to: {CONFIG["output_path"]}')
print(f'Total columns: {len(df_output.columns)} (added {len(new_cols)} new)')
for col in new_cols:
    print(f'  + {col}')


In [ ]:
# Cell 12: Save Manifest

manifest = {
    'notebook': '07_finbert_distress_score',
    'mode': 'zero_shot',
    'output_file': CONFIG['output_path'],
    'model_checkpoint': str(best_model_path),
    'pretrained_model': CONFIG['pretrained_model'],
    'distress_score_cols': ['distress_turn_1', 'distress_turn_2', 'distress_turn_3',
                            'distress_avg', 'distress_max', 'distress_shift'],
    'distress_score_formula': 'P(negative)*1.0 + P(neutral)*0.5 + P(positive)*0.0',
    'label_mapping': {str(k): v for k, v in model.config.id2label.items()},
    'model_config': {
        'max_length': CONFIG['max_length'],
        'scoring': 'zero-shot classifier probabilities',
    },
    'training_summary': {
        'fine_tuned': False,
        'reason': 'zero-shot mode — ROCm GPU incompatibility on Radeon 8060S',
    },
    'split_source': CONFIG['bilstm_split_path'],
}

with open(CONFIG['manifest_path'], 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Manifest saved to: {CONFIG["manifest_path"]}')
print('\n' + '=' * 50)
print('NOTEBOOK 07 COMPLETE')
print('=' * 50)

In [ ]:
# Cell 13: Sanity Check

print('SANITY CHECK')
print('=' * 50)

df_check = pd.read_csv(CONFIG['output_path'])

distress_cols = ['distress_turn_1', 'distress_turn_2', 'distress_turn_3',
                 'distress_avg', 'distress_max', 'distress_shift']

print(f'1. Output readable:          OK - Shape {df_check.shape}')
print(f'2. Distress cols present:    {"OK" if all(c in df_check.columns for c in distress_cols) else "FAIL"}')
print(f'3. No NaN in distress cols:  {"OK" if df_check[distress_cols].isna().sum().sum() == 0 else "FAIL"}')
print(f'4. Scores in [0,1] range:    {"OK" if df_check["distress_avg"].between(0, 1).all() else "FAIL"}')
print(f'5. customer_id present:      {"OK" if "customer_id" in df_check.columns else "FAIL"}')
print(f'6. Shift has both signs:     {"OK" if (df_check["distress_shift"] > 0).any() and (df_check["distress_shift"] < 0).any() else "FAIL"}')

if 'distress_encoded' in df_check.columns:
    corr = df_check['distress_avg'].corr(df_check['distress_encoded'])
    print(f'7. Avg score vs encoded corr: {corr:.4f}')

print('\n' + '=' * 50)
print('Ready for Notebook 08 (XGBoost)!')
print('=' * 50)
